In [ ]:
import sys
print('Python: {}'.format(sys.version))
import scipy
print('scipy: {}'.format(scipy.__version__))
import numpy
print('numpy: {}'.format(numpy.__version__))
import matplotlib
print('matplotlib: {}'.format(matplotlib.__version__))
import pandas as pd
import sklearn
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.gaussian_process.kernels import RBF
from sklearn.inspection import DecisionBoundaryDisplay
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.svm import SVC
import matplotlib.pyplot as plt
import numpy as np
from sklearn import datasets
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.gaussian_process.kernels import RBF
from sklearn.inspection import DecisionBoundaryDisplay
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
import seaborn as sns
from sklearn.metrics import classification_report
plt.rcParams["figure.dpi"] = 1200

In [ ]:
def loadData(fileName):
  return pd.read_excel(fileName)

In [ ]:
def loadClassifiers():
  C = 10
  kernel = 1.0 * RBF([1.0, 1.0])  # for GPC
  return {
    "Gaussian NB": GaussianNB(),
    "Logistic Regression": LogisticRegression(multi_class="multinomial"),
    'L1 Logistic': LogisticRegression( penalty='l1', solver='saga', multi_class='multinomial'),
    'L2 Logistic ': LogisticRegression(penalty='l2', solver='saga', multi_class='multinomial'),
    "Random Forest": RandomForestClassifier(),
    }

In [ ]:
def dataClassify(data):
  return data.drop(columns=['RevenueLossbyUpgrade', 'RevenuebyStays', 'RevenueLossbyCancel','PotentialRevenue','ArrivalDate','ReservationStatus','ReservationStatusDate','CustomerType'])

In [ ]:
def dimReduction(X_train):
  pca = PCA(n_components='mle')
  pca.fit(X_train)
  print('number of components: ' ,pca.n_components_)
  print('explained variance: \n',pca.explained_variance_)
  print('explained variance ratio: \n',np.cumsum(pca.explained_variance_ratio_))
  return pca.fit_transform(X_train)

In [ ]:
def getTrainSet(data):
  dataSet=dataClassify(data)
  Y_train=dataSet[['AssignedRoom']]
  X_train=dataSet.drop(columns=['AssignedRoom'])
  catlabels=['BookingDistributionChannel','Country','DepositType','MarketSegment','BookingMeal',]
  #numlabels=['AverageDailyRate','BookingChanges','BookingParking','DaysInWaitingList','LeadTime','NumberOfAdult','NumberOfBabies','NumberOfChildren','PreviousBookingsNotCanceled','StaysInWeekendNights','StaysInWeekNights','TotalOfSpecialRequests']
  for i in catlabels:
    X_train[i] = X_train[i].astype('category')
  cat_columns = X_train.select_dtypes(['category']).columns
  X_train[cat_columns] = X_train[cat_columns].apply(lambda x: x.cat.codes)
  tmp=[]
  for i in X_train['ReservedRoom']:
    tmp.append(ord(i)-65)
  X_train['ReservedRoom']=tmp
  tmp=[]
  for i in Y_train['AssignedRoom']:
    tmp.append(ord(i)-65)
  Y_train['AssignedRoom']=tmp
  #return [dimReduction(X_train.to_numpy()),list(Y_train['AssignedRoom'])]
  return [X_train.to_numpy(),list(Y_train['AssignedRoom'])]

In [ ]:
def classificationReport(true,pred,target_names,title,ax):
  clf_report = classification_report(true,pred,labels=np.arange(7),target_names=target_names,output_dict=True)
  ax.set_title(title)
  sns.heatmap(pd.DataFrame(clf_report).iloc[:-1, :].T, annot=True,ax=ax)

In [ ]:
def fitClassify(x,y,classifiers):
  fitted=[]
  N=len(classifiers)
  fig, axes = plt.subplots(1, N, sharex=True,figsize=(21,15))
  fig.suptitle('Classifiers Comparison')
  count=0
  for classifier_idx, (name, classifier) in enumerate(classifiers.items()):
    tmp=classifier.fit(x, y)
    fitted.append(tmp)
    y_pred = tmp.predict(x)
    accuracy = accuracy_score(y, y_pred)
    print(f"Accuracy (train) for {name}: {accuracy:0.1%}")
    classificationReport(y,y_pred,[chr(i+65) for i in tmp.classes_],name,axes[count])
    count+=1
  plt.savefig("Upgrade Model.png",transparent=True)
  plt.show()
  return fitted

In [ ]:
def getProba(bestFitted,x):
  to_return=pd.DataFrame(bestFitted.predict_proba(x),columns =[chr(i+65) for i in bestFitted.classes_])
  to_return.to_excel("Upgrade_proba.xlsx",index=False)
  return to_return

In [ ]:
def lossMatrix(dataset,ADRbyRoom):
  lossmatrix=numpy.empty([0,7])
  for row in dataset[['ArrivalWeekNumber','ReservedRoom','AverageDailyRate','StaysInWeekendNights','StaysInWeekNights']].iterrows():
    tmp=(ADRbyRoom.loc[(row[1]['ArrivalWeekNumber'],),'ADR'].to_numpy()-row[1]['AverageDailyRate'])*(row[1]['StaysInWeekendNights']+row[1]['StaysInWeekNights'])
    tmp[tmp<0]=0
    tmp[ord(row[1]['ReservedRoom'])-65]=0
    lossmatrix=np.vstack((lossmatrix,tmp))
  pd.DataFrame(lossmatrix,columns =['A','B','C','D','E','F','G']).to_excel("Lossmatrix.xlsx",index=False)
  return lossmatrix

In [ ]:
data=loadData('R2_final_hotels_Clean.xlsx')

In [ ]:
classTrainSet=getTrainSet(data)

In [ ]:
fittedModel=fitClassify(classTrainSet[0],classTrainSet[1],loadClassifiers())

In [ ]:
getProba(fittedModel[-1],classTrainSet[0])

In [ ]:
ADRbyRoom=pd.read_excel("ADRbyRoom&Week.xlsx").groupby(['ArrivalWeekNumber','ReservedRoom']).sum(['ADR'])

In [ ]:
lossMatrix(data,ADRbyRoom)

In [ ]:
upgradeLossMatrix=pd.read_excel("Lossmatrix.xlsx").to_numpy().T

In [ ]:
probaMatrix=pd.read_excel("Upgrade_proba.xlsx").to_numpy()

In [ ]:
def calUpgradePpremium(upgradeLossMatrix,probaMatrix,dataset):
  upgradepremium=[]
  for i in range(probaMatrix.shape[0]):
    upgradepremium.append(np.dot(probaMatrix[i,:],upgradeLossMatrix[:,i]))
  dataset['UpgradePremium']=upgradepremium
  dataset['UpgradePremium']=dataset['UpgradePremium']/(dataset['StaysInWeekendNights']+dataset['StaysInWeekNights'])
  dataset['UpgradePremium']=dataset['UpgradePremium'].replace(np.nan, 0)
  return dataset

In [ ]:
data_upgradepremium=calUpgradePpremium(upgradeLossMatrix,probaMatrix,data)

In [ ]:
data_upgradepremium.to_excel("R2_final_hotels_upgradepremium.xlsx",index=False)